# Notebook 13 — Fine-Tuning & RLHF: Shaping Model Behavior
**AI Models and Agents for Business · Week 7 · Session 13**

Every notebook so far has used a model *as-is* — you changed the prompt, not the model. Today you change the model itself, two ways:

| Part | What you'll do | Real technique, scaled down |
|---|---|---|
| A | **Fine-tuning** — retrain a tiny model on a small, brand-specific dataset | Yes — a genuine training run, just on a model small enough to train on a laptop in minutes instead of a data center for weeks |
| B | **RLHF, in miniature** — train a reward model on human-rated responses, then use it to score and rank candidate responses | Partly — a toy version of the reward-modeling idea behind RLHF, without the reinforcement-learning loop that normally sits on top of it |

**APIs/libraries used:** `transformers` (local, no key) for Part A; `sentence-transformers` + `scikit-learn` (both already familiar from Notebooks 2 and 5) + Groq for Part B.

**⚠️ Time budget:** Part A's training cell (Cell A4) can take 1–4 minutes depending on classroom wifi/compute — see the note there. If your section is running short on time, this is a reasonable one to run as an instructor demo rather than everyone running it in parallel, the same call already made for the Ollama demo.


---
## Part A: Fine-Tuning a Tiny Model

**The idea:** every model you've used so far (Groq's gpt-oss, Claude, GPT) was pretrained once on a huge, general-purpose dataset. Fine-tuning takes that pretrained model and keeps training it — for a little while longer, on a much smaller, specific dataset — to shift its behavior toward *your* data. Same underlying mechanism whether it's a $10M enterprise fine-tune or the toy version you're about to run: gradient descent, just on new examples.

We'll fine-tune `distilgpt2` (82 million parameters — tiny by modern standards, which is exactly why it can train on a CPU in class) to write in the voice of a fictional bank's customer support team, **Nimbus Bank**.

## Cell A1 — Setup

In [ ]:
!pip install transformers datasets torch --quiet

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

MODEL_NAME = "distilgpt2"  # 82M parameters — small enough to fine-tune on CPU in a few minutes

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print(f"Loaded {MODEL_NAME}: {base_model.num_parameters():,} parameters")

## Cell A2 — Baseline: what the untrained voice sounds like

Before any fine-tuning, ask the base model to continue a Nimbus Bank support reply. It has never heard of Nimbus Bank — watch it either go generic or drift off-topic.

In [ ]:
def generate(model, prompt, max_new_tokens=40, seed=0):
    # Fixing the random seed means two models get the same random draws,
    # so a difference in output comes from the weights, not from luck.
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=0.8, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_PROMPT = "Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your late fee,"

print("=== BEFORE fine-tuning ===")
print(generate(base_model, TEST_PROMPT))

## Cell A3 — A tiny brand-voice dataset

Real fine-tuning datasets run from hundreds to millions of examples. This one has 16 — enough to *see* the shift, nowhere near enough to trust in production. That gap is itself the lesson (bridge to Cell A6's discussion).

**✏️ CHANGE ME** — edit a few of these to see how the fine-tuned voice changes.

In [ ]:
nimbus_examples = [
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your late fee, we've reviewed your account and issued a full waiver as a one-time courtesy.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your late fee, our records show it was applied correctly per your account terms, though we're happy to discuss a payment plan.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your card dispute, we've opened a formal investigation and you'll hear back within 5 business days.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your overdraft, we've temporarily paused further fees while we review your account together.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your loan application, we're pleased to let you know it has moved to final underwriting.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your closed account, we can confirm the closure was processed and no further fees will apply.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your wire transfer, it has been sent and should arrive within one to two business days.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your interest rate, we've applied the promotional rate you qualified for at account opening.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your fraud alert, we've frozen the card as a precaution and a new one is on its way.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your statement request, we've attached the last three months to this message.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your mobile deposit, it has cleared and the funds are now available.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your credit limit increase, we've approved a modest increase effective immediately.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your billing error, we've corrected the charge and refunded the difference.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your account access, we've reset your login and sent new instructions to your email.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your recurring payment, we've cancelled it per your request effective this cycle.",
"Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your check deposit, it will be fully available in your account within two business days.",
]

train_dataset = Dataset.from_dict({"text": nimbus_examples})

def tokenize(batch):
    # Add an end-of-text token so the model learns where a reply stops
    texts = [t + tokenizer.eos_token for t in batch["text"]]
    out = tokenizer(texts, truncation=True, padding="max_length", max_length=64)
    # "labels" are what the model is graded on. Padding positions get -100, which means
    # "don't grade this", so the model isn't trained to predict filler.
    out["labels"] = [[tok if keep == 1 else -100 for tok, keep in zip(ids, mask)]
                     for ids, mask in zip(out["input_ids"], out["attention_mask"])]
    return out

tokenized_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"{len(train_dataset)} training examples, tokenized.")

## Cell A4 — Fine-tune, live ← the actual training run

This is a real training loop — the same `Trainer` API used for production fine-tunes, just pointed at 16 examples and 3 epochs instead of millions of examples and days of GPU time. **Expect 1–4 minutes.** Loss should visibly drop across epochs; if it doesn't move at all, that's worth flagging in class.

In [ ]:
training_args = TrainingArguments(
    output_dir="./nimbus_finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=2,
    save_strategy="no",
    report_to=[],
)

fine_tuned_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

trainer = Trainer(
    model=fine_tuned_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()
print("\nFine-tuning complete.")

## Cell A5 — Before vs. after, side by side

Same prompt, same decoding settings, same random seed, two different sets of weights. Sampling is random, so we draw **three** samples from each model: one lucky draw shouldn't convince you.

In [ ]:
for seed in [0, 1, 2]:
    print(f"--- Sample {seed + 1} (same seed for both models) ---")
    print("BEFORE (base distilgpt2):", generate(base_model, TEST_PROMPT, seed=seed))
    print("AFTER  (fine-tuned)     :", generate(fine_tuned_model, TEST_PROMPT, seed=seed))
    print()

print()
print("Try a prompt the training set never saw exactly:")
NEW_PROMPT = "Dear valued customer, thank you for reaching out to Nimbus Bank. Regarding your savings account,"
print(generate(fine_tuned_model, NEW_PROMPT))

### 💬 Discussion — Cell A6
- Did the fine-tuned model pick up the *phrasing* ("Dear valued customer... thank you for reaching out"), the *topics*, or both? Which is easier to learn from 16 examples?
- Run the untrained-topic prompt (savings account) with a few different seeds, e.g. `generate(fine_tuned_model, NEW_PROMPT, seed=5)`. Does it generalize sensibly, or does it start to ramble? What does that tell you about how much data 16 examples really is? (This is overfitting, which you will measure directly in Session 22: memorizing a small training set instead of learning the pattern.)
- **Business framing:** a company with 50,000 real support transcripts could fine-tune something like this to sound genuinely on-brand. A company with 16 examples — like you just did — mostly teaches the model to memorize, not generalize. Where's the line, and how would you find out where it is before shipping this to real customers?
- **Fine-tune vs. prompt vs. RAG:** you could also have gotten "sounds like Nimbus Bank" by putting a few examples in a system prompt (few-shot prompting, Session 16) or by retrieving real past tickets (RAG, Sessions 17-18). What does fine-tuning buy you that those don't — and what does it cost that they don't (retraining every time the brand voice changes, no built-in citation trail, GPU/data requirements)? This is the build-vs-buy lens you'll use in Session 20, one level more technical.

### ✅ Concept check — fine-tuning
Answer each one in your head, or out loud with your partner, *before* you open the answer. If you miss one, re-read that part before moving on.

**1. What does fine-tuning change: the prompt, the model's weights, or the documents it can look up?**
<details><summary>Check your answer</summary>

The **weights**. Prompting changes the input; RAG changes what documents are pasted into the input; fine-tuning changes the model itself.
</details>

**2. The fine-tuned model nails the Nimbus phrasing on topics from the training set but rambles on a new topic. What is the likely cause, and what would you need before trusting it?**
<details><summary>Check your answer</summary>

With 16 examples it has largely **memorized** them rather than learned a general pattern (overfitting). You'd need many more, more varied examples, and a held-out set of prompts it never trained on to check that it generalizes.
</details>

**3. Why did we fix the random seed before comparing the base and fine-tuned models?**
<details><summary>Check your answer</summary>

Generation samples randomly. With the same seed, both models get the same random draws, so a difference in output comes from the **weights**, not from luck.
</details>

---
## Part B: RLHF in Miniature — Reward Models & Reward Hacking

**The idea:** RLHF (Reinforcement Learning from Human Feedback) is how a raw, pretrained model gets shaped into something like Claude or ChatGPT — helpful, on-tone, willing to say "I don't know." The real pipeline has three stages: (1) humans rank candidate outputs, (2) a **reward model** learns to predict those rankings, (3) the base model is nudged, via reinforcement learning, toward outputs the reward model scores highly.

Stage 3 (the reinforcement-learning loop itself, PPO) is genuinely out of scope for a single class session. Stages 1 and 2 — the part that actually encodes *what humans wanted* — are not, and that's what you'll build.

## Cell B1 — Setup (a small embedding model plus a Groq call)

In [ ]:
!pip install sentence-transformers groq scikit-learn --quiet

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from groq import Groq
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # the same embedding model the RAG sessions (17-18) use

# Your Groq key comes from Colab Secrets (key icon on the left, name GROQ_API_KEY, Notebook access on).
# If it is not there, you will be asked to paste it in a box that hides what you type.
def _get_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

GROQ_API_KEY = _get_key()
client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-20b"

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

print("Ready.")

## Cell B2 — A tiny human-preference dataset

In real RLHF, thousands of human raters compare **pairs** of model outputs side by side and mark which one they prefer, and the reward model learns from those comparisons. To keep this simple, we skip the pairing: here are 10 customer-service responses a manager rated **one at a time**, for the same "angry customer, refund outside policy" scenario, each marked preferred (`1`) or not (`0`). That's a simplification of how real reward models are trained.

In [ ]:
rated_responses = [
    ("I completely understand your frustration, and while this falls outside our standard refund window, let me see what options I can offer you today.", 1),
    ("Per policy section 4.2, refunds outside the 30-day window are not permitted under any circumstances.", 0),
    ("That sounds incredibly frustrating — I'm sorry you're dealing with this. Let's figure out a path forward together.", 1),
    ("Unfortunately there is nothing further I am able to do for you regarding this matter.", 0),
    ("I hear you, and I want to make this right. I can offer a partial credit even though it's past the window — would that help?", 1),
    ("This has been escalated. Please wait 5-7 business days for a response with no further updates in the meantime.", 0),
    ("I'm really sorry this happened. Here's exactly what I can do right now, and here's what I'll follow up on by tomorrow.", 1),
    ("As previously stated, our policy does not allow for exceptions of this nature.", 0),
    ("Thanks for your patience — I've found a way to get you a partial refund today even though we're past the standard window.", 1),
    ("I am not able to assist further. Please contact our general support line.", 0),
]

texts = [t for t, label in rated_responses]
labels = [label for t, label in rated_responses]
print(f"{len(texts)} rated examples, {sum(labels)} marked 'preferred'.")

## Cell B3 — Train the reward model

This is, quite literally, what "reward model" means: a classifier that takes text in and predicts a human-preference score out. Real reward models are themselves large neural networks trained on millions of comparisons; this is a logistic regression trained on ten, which makes it fast, transparent, and — as you'll see in Cell B6 — easy to fool.

In [ ]:
X_train = embedder.encode(texts)
reward_model = LogisticRegression()
reward_model.fit(X_train, labels)

def reward_score(text):
    """Returns a 0-1 'human would prefer this' probability."""
    emb = embedder.encode([text])
    return reward_model.predict_proba(emb)[0][1]

for t, label in rated_responses[:4]:
    print(f"true={label}  reward_score={reward_score(t):.2f}  {t[:60]}...")

## Cell B4 — Generate candidate responses with Groq

Ask the model for the same reply five times at a fairly high temperature, so the candidates actually differ — this simulates the "sample many candidate outputs" step that sits upstream of reward scoring in a real RLHF pipeline.

In [ ]:
SCENARIO = """A customer is angry that a $340 charge wasn't refunded because it's 3 days
past your company's 30-day refund window. Write a short reply as the support agent."""

candidates = []
for i in range(5):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": SCENARIO}],
        max_tokens=120,
        temperature=0.9,
    )
    candidates.append(r.choices[0].message.content)

for i, c in enumerate(candidates):
    print(f"\n--- Candidate {i+1} ---\n{c}")

## Cell B5 — Score and rank the candidates ← this is the RLHF mechanism, in one pass

In the real pipeline, this reward score is what the reinforcement-learning step (PPO) would use to nudge the *model's own weights* toward more responses like the top-ranked one, repeated over millions of examples. Here, we just do the scoring and ranking — the piece that encodes human preference — without the retraining loop on top.

In [ ]:
scored = sorted(
    [(reward_score(c), c) for c in candidates],
    key=lambda x: x[0], reverse=True
)

print("Ranked by reward model (highest = most 'human-preferred'):\n")
for rank, (score, text) in enumerate(scored, 1):
    print(f"#{rank}  score={score:.2f}")
    print(f"   {text}\n")

### 💬 Discussion — Cell B5
- Does the top-ranked candidate actually match what *you'd* consider the best response? Where does the reward model agree with your own judgment, and where does it miss?
- Look back at the five "preferred" responses in Cell B2. Three of them offer money past the 30-day window. Did the rater reward **empathy**, or **concessions**? If company policy allows no exceptions, what has the reward model just learned to favor? How would you rewrite the rating rubric so that policy compliance counts?
- The reward model only ever saw 10 examples about one scenario. Would you trust it to score a completely different kind of customer complaint?

## Cell B6 — Reward hacking, live ← the key experiment

Real reward models can be gamed — a model being RL-trained will happily find the loophole in whatever the reward model actually measures, not what it was *supposed* to measure. That failure mode is called **reward hacking**, and it's one of the clearest, least abstract entry points into the alignment/safety territory this course returns to in Session 24.

Below, we build a deliberately naive reward model — one trained only on response *length*, nothing about actual helpfulness — and show it can be fooled by a long, empty response.

In [ ]:
# A "lazy" reward model that only rewards length, standing in for
# any reward signal that measures a PROXY for quality instead of quality itself
def naive_length_reward(text):
    return min(len(text) / 500, 1.0)  # longer = "better", capped at 1.0

# A deliberately padded, low-substance response: long, apologetic, resolves nothing
padded_response = (
    "I am so very sorry to hear about this experience, and I truly want you to know "
    "that your feedback means a great deal to us here at the company. We take every "
    "customer concern seriously and are always working to improve. I understand how "
    "frustrating this must feel, and I want to assure you that we value you as a "
    "customer. Please know that we appreciate your patience and understanding during "
    "this time, and we are grateful for the opportunity to serve you."
)

# A short, genuinely useful response that actually resolves the issue
concise_response = "I can issue a partial refund today even though we're past the window — does that work for you?"

print(f"Naive (length-only) reward — padded, empty response: {naive_length_reward(padded_response):.2f}")
print(f"Naive (length-only) reward — concise, useful response: {naive_length_reward(concise_response):.2f}")
print()
print(f"Your trained reward model  — padded, empty response: {reward_score(padded_response):.2f}")
print(f"Your trained reward model  — concise, useful response: {reward_score(concise_response):.2f}")

### 💬 Discussion — Cell B6
- The naive length-based reward model scores the padded, empty response *higher* than the genuinely useful one. If this were the actual reward signal driving RL training, what would the model learn to do more of over millions of updates?
- Your Cell B3 reward model was trained on real preference labels, not just length — does it also get fooled, even a little? What would you need to check before trusting a reward model in production?
- Connect this to Session 5's Amazon case: that model wasn't "wrong" by its own metric either — it optimized exactly what it was trained to optimize, and the metric was the problem. Same failure shape, different mechanism.

### ✅ Concept check — reward models
Answer each one in your head, or out loud with your partner, *before* you open the answer. If you miss one, re-read that part before moving on.

**1. What does a reward model take in, and what does it put out?**
<details><summary>Check your answer</summary>

It takes in a piece of text and puts out a **score**: a prediction of how much a human rater would prefer it. On its own it doesn't write anything or change the main model.
</details>

**2. In one sentence, what is reward hacking?**
<details><summary>Check your answer</summary>

Getting a high score on the **measurable proxy** (here, length) without delivering the real goal (a helpful, policy-compliant reply).
</details>

**3. Name two things real RLHF has that Part B left out.**
<details><summary>Check your answer</summary>

Raters comparing **pairs** of responses at large scale, and the **reinforcement-learning step** that uses the reward model's scores to update the main model's weights.
</details>

---
## Model Swap: Anthropic Claude or OpenAI GPT-4o

Part B's candidate-generation call (Cell B4) works identically on any provider — only the client and response-parsing syntax change.

In [ ]:
# MODEL SWAP — Anthropic Claude
# Uncomment if you have an Anthropic key

# !pip install anthropic --quiet
# import anthropic
# ANTHROPIC_KEY = "sk-ant-PASTE_YOUR_KEY_HERE"
# claude = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
#
# result = claude.messages.create(
#     model="claude-3-5-haiku-20241022",
#     max_tokens=120,
#     temperature=0.9,
#     messages=[{"role": "user", "content": SCENARIO}]
# )
# print(result.content[0].text)

# MODEL SWAP — OpenAI GPT-4o
# Uncomment if you have an OpenAI key

# !pip install openai --quiet
# from openai import OpenAI
# OPENAI_KEY = "sk-PASTE_YOUR_KEY_HERE"
# oai = OpenAI(api_key=OPENAI_KEY)
#
# result = oai.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": SCENARIO}],
#     max_tokens=120,
# )
# print(result.choices[0].message.content)

# Note: Claude's response text lives at .content[0].text; Groq and OpenAI both
# use .choices[0].message.content — this is the same API difference flagged
# in every other notebook's model-swap section.

print("Uncomment one of the blocks above to compare models.")

## ✅ Notebook 13 complete — Reflection questions

These prep you for **Quiz 4** (Session 16).

1. What did fine-tuning change about the model's behavior in Part A that prompting alone couldn't have changed?
2. With only 16 training examples, how would you know whether the fine-tuned model *learned the pattern* versus *memorized the examples*? What would you check?
3. In your own words, what does a reward model actually predict, and what does it *not* do on its own (i.e., what's still missing from a full RLHF pipeline)?
4. Give one example — not from class — of a business metric that could be "reward hacked" the way the naive length-based reward model was in Cell B6.
5. If your team's project used a fine-tuned or RLHF-style model, what evidence would you want before trusting its outputs with a real customer?

---
*Next session: Reasoning Models. Today you changed how a model behaves by changing its weights; next class is about letting it spend more computation thinking before it answers, and what that costs. (Who should be changing model weights at all, you, an open-source community, or a vendor, is the Build vs. Buy question in Session 20.)*